# MuseTalk 아바타 추론 서버 (Colab GPU)

이 노트북은 **MuseTalk 전용 추론 서버**입니다. 로컬 백엔드(FastAPI)가 이 노트북에 오디오를 보내면, 립싱크된 아바타 영상을 반환합니다.

## 왜 conda 가상환경을 따로 쓰는가
Colab 기본 런타임은 Python 3.12인데, MuseTalk이 요구하는 numpy/torch/mmcv 등 구버전 패키지들은 Python 3.12용 wheel이 없어 그대로 설치하면 실패합니다. "런타임 버전" 드롭다운은 Python 버전을 고르는 게 아니라 백엔드 이미지 스냅샷 날짜를 고르는 것이라 이 문제를 해결해주지 못합니다.
그래서 노트북 안에 **Miniconda로 Python 3.10 환경을 하나 더 만들고**, 검증된 버전 조합(로컬 MoodTender 프로젝트에서 실제 동작 확인된 조합)을 그 환경에 설치합니다. MuseTalk 서버는 그 환경의 파이썬으로 별도 프로세스로 실행되고, 노트북 커널(3.12)은 그 프로세스에 ngrok 터널만 연결해주는 역할만 합니다. (가상환경/별도 프로세스라고 해서 GPU 추론 속도가 느려지지 않습니다 — CUDA 접근 경로는 동일합니다.)

conda 환경 전체(4-1번 셀)와 체크포인트(5번 셀), 아바타 준비 결과는 모두 Drive에 캐싱되므로, 런타임이 끊겨도 재설치 없이 몇 분 안에 복구됩니다.

## 시작 전 체크리스트
1. **런타임 > 런타임 유형 변경**에서 GPU를 `A100` 또는 `L4`로 설정했는지 확인
2. **왼쪽 사이드바 열쇠(🔑) 아이콘 (Secrets)**에 아래 3개를 등록하고, 각각 "노트북 액세스" 토글을 켜주세요. (토큰을 노트북 파일에 직접 적으면 git에 커밋될 때 유출될 수 있어 이 방식으로 관리합니다)
   - `HF_TOKEN` — https://huggingface.co/settings/tokens 에서 발급 (Read 권한)
   - `NGROK_AUTHTOKEN` — https://dashboard.ngrok.com/get-started/your-authtoken 에서 발급
   - `NGROK_STATIC_DOMAIN` — https://dashboard.ngrok.com/domains 에서 예약한 무료 고정 도메인 (예: your-name.ngrok-free.app)
3. 면접관 아바타로 쓸 5초 영상 파일을 Google Drive의 `MyDrive/musetalk_assets/avatar_source.mp4` 경로에 업로드

## 노트북 실행 순서
1~8번 셀: 환경 구축 (conda 생성/복원, repo 클론, 패키지 설치+Drive 캐싱, 체크포인트/아바타 서버 스크립트 준비) — 최초 1회만 오래 걸리고 이후엔 Drive 캐시로 빨라짐
9~12번 셀: 서버를 subprocess로 기동 + 로컬 루프백으로 순수 추론 속도 벤치마크
13번 셀: ngrok으로 외부 노출

In [ ]:
# 1. GPU 확인 + Google Drive 마운트
!nvidia-smi --query-gpu=name,memory.total --format=csv

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Miniconda 설치 + Python 3.10 가상환경 생성 (Drive에 캐싱된 게 있으면 복원만 함)
# Colab 기본 런타임(3.12)과 별개로, MuseTalk 실행 전용 Python 3.10 환경을 만듭니다.
import os

CONDA_PREFIX = "/usr/local/miniconda"
ENV_NAME = "musetalk"
CONDA_BIN = f"{CONDA_PREFIX}/bin/conda"
CONDA_PY = f"{CONDA_PREFIX}/envs/{ENV_NAME}/bin/python"
CONDA_PIP = f"{CONDA_PREFIX}/envs/{ENV_NAME}/bin/pip"

DRIVE_CONDA_CACHE = "/content/drive/MyDrive/musetalk_conda_cache/musetalk_conda_env.tar"

if not os.path.exists(CONDA_PY):
    if os.path.exists(DRIVE_CONDA_CACHE):
        print("[캐시 발견] Drive에서 conda 환경 전체를 복원합니다 (몇 분 소요)...")
        os.makedirs(CONDA_PREFIX, exist_ok=True)
        !tar -xf "{DRIVE_CONDA_CACHE}" -C {CONDA_PREFIX}
    else:
        print("[캐시 없음] Miniconda 최초 설치를 진행합니다...")
        if not os.path.exists(CONDA_BIN):
            !wget -O /content/miniconda.sh https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh
            # -u: 대상 폴더가 이미 있어도(이전 시도 잔여물) 갱신 설치로 처리
            !bash /content/miniconda.sh -b -u -p {CONDA_PREFIX}
        # 최신 Anaconda 기본 채널(pkgs/main, pkgs/r)은 비대화형 설치 전에 이용약관 동의가 필요합니다 (최초 1회).
        !{CONDA_BIN} tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
        !{CONDA_BIN} tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r
        !{CONDA_BIN} create -y -n {ENV_NAME} python=3.10
else:
    print("conda 환경이 이미 준비되어 있습니다:", CONDA_PY)

# 셸 명령이 중간에 실패해도 파이썬 예외가 나지 않으므로, 실제로 준비됐는지 직접 확인합니다.
if os.path.exists(CONDA_PY):
    print("conda 환경 준비 완료:", CONDA_PY)
else:
    print("conda 환경 준비 실패. 아래 진단 정보를 확인하세요.")
    print("- conda 설치 여부:", os.path.exists(CONDA_BIN))
    !ls {CONDA_PREFIX}/envs 2>/dev/null || echo "envs 폴더 자체가 없음"
    raise RuntimeError("conda 환경이 생성되지 않았습니다. 위 로그를 보고 원인을 확인한 뒤 이 셀을 다시 실행하세요.")

# 노트북 커널(3.12) 쪽에서 필요한 도구 (ngrok 터널, 루프백 벤치마크용)
!pip install -q pyngrok websockets

In [ ]:
# 3. MuseTalk 레포 클론 (최초 1회)
# "폴더 존재 여부"가 아니라 "musetalk 소스 코드 존재 여부"로 판단합니다.
# (data/models 폴더는 체크포인트 다운로드 단계에서 os.makedirs로 미리 생겼을 수 있어,
# 폴더 존재만으로는 클론 완료 여부를 알 수 없습니다.)
%cd /content

if os.path.exists("/content/MuseTalk/musetalk"):
    print("MuseTalk 레포가 이미 준비되어 있습니다.")
elif not os.path.exists("/content/MuseTalk"):
    print("MuseTalk 폴더가 없어 새로 클론합니다...")
    !git clone https://github.com/TMElyralab/MuseTalk.git
else:
    print("MuseTalk 폴더는 있지만 레포 소스가 없습니다 (체크포인트만 먼저 받아진 상태).")
    print("기존 폴더(체크포인트 등)는 유지한 채 레포 소스만 받아옵니다...")
    %cd /content/MuseTalk
    !git init -q
    !git remote add origin https://github.com/TMElyralab/MuseTalk.git 2>/dev/null || git remote set-url origin https://github.com/TMElyralab/MuseTalk.git
    !git fetch --depth 1 origin main
    !git checkout -f main
    %cd /content

%cd /content/MuseTalk
print("musetalk 패키지 존재 여부:", os.path.exists("/content/MuseTalk/musetalk"))

In [ ]:
# 4. 검증된 버전 조합으로 conda 환경에 설치
# 아래 목록은 로컬(Windows, MoodTender 프로젝트)에서 실제로 동작을 확인한 조합입니다.
# tensorrt/pywin32/pyreadline3 등 Windows 전용 패키지는 Linux(Colab)와 무관하므로 제외했습니다.
# mmengine/mmcv/mmdet/mmpose는 mim으로 따로 설치합니다 (플랫폼별 prebuilt wheel을 정확히 찾아주기 때문).
# chumpy는 별도로 --no-build-isolation 설치합니다 (아래 참고).

requirements_text = """
numpy==1.23.5
basicsr==1.4.2
gfpgan==1.3.8
facexlib==0.3.0
diffusers==0.30.2
transformers==4.39.2
accelerate==0.28.0
huggingface-hub==0.30.2
tokenizers==0.15.2
safetensors==0.7.0
opencv-python==4.9.0.80
librosa==0.11.0
soundfile==0.12.1
ffmpeg-python==0.2.0
imageio==2.37.3
imageio-ffmpeg==0.6.0
moviepy==1.0.3
einops==0.8.1
omegaconf==2.3.0
scikit-learn==1.7.2
scipy==1.15.3
numba==0.65.1
tqdm==4.65.2
PyYAML==6.0.3
openai-whisper==20250625
fastapi==0.136.3
uvicorn==0.48.0
websockets==15.0.1
"""

with open("/content/musetalk_requirements.txt", "w") as f:
    f.write(requirements_text)

# torch는 cu118 전용 인덱스에서 받아야 해서 별도 설치
!{CONDA_PIP} install -q --no-cache-dir --index-url https://download.pytorch.org/whl/cu118 torch==2.0.1+cu118 torchvision==0.15.2+cu118 torchaudio==2.0.2+cu118

# numpy를 먼저 설치해둔 뒤, chumpy는 빌드 격리를 꺼서 이 numpy를 그대로 쓰게 합니다.
# (빌드 격리를 켠 채로 두면 chumpy가 격리된 새 환경에서 최신 numpy를 따로 받다가 깨집니다)
!{CONDA_PIP} install -q numpy==1.23.5
!{CONDA_PIP} install -q --no-build-isolation chumpy==0.70

!{CONDA_PIP} install -q -r /content/musetalk_requirements.txt

# OpenMMLab 계열은 mim으로 설치 (버전은 MoodTender에서 검증된 것과 동일)
!{CONDA_PIP} install -q --no-cache-dir -U openmim
!{CONDA_PY} -m mim install -q mmengine
!{CONDA_PY} -m mim install -q "mmcv==2.0.1"
!{CONDA_PY} -m mim install -q "mmdet==3.1.0"
!{CONDA_PY} -m mim install -q "mmpose==1.1.0"

# mim install 계열이 버전 고정 없이 opencv-python/numpy를 최신으로 끌어올리는 경우가 있어
# 마지막에 다시 원하는 버전으로 강제 고정합니다 (numpy 2.x가 들어오면 여러 패키지가 ABI 충돌로 깨짐).
!{CONDA_PIP} install -q "numpy==1.23.5" "opencv-python==4.9.0.80"

print("conda 환경 설치 완료 - 버전 확인:")
!{CONDA_PY} -c "import numpy, cv2, torch; print('numpy', numpy.__version__); print('opencv-python', cv2.__version__); print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())"

In [ ]:
# 4-1. conda 환경 전체를 Drive에 캐싱 (한 번만 하면 됨 - 다음 런타임부터는 2번 셀에서 자동 복원)
if not os.path.exists(DRIVE_CONDA_CACHE):
    print("conda 환경을 Drive에 캐싱합니다 (용량에 따라 몇 분 소요될 수 있습니다)...")
    !tar -cf /content/musetalk_conda_env.tar -C {CONDA_PREFIX} .
    os.makedirs(os.path.dirname(DRIVE_CONDA_CACHE), exist_ok=True)
    !cp /content/musetalk_conda_env.tar "{DRIVE_CONDA_CACHE}"
    print("Drive 캐싱 완료 - 다음부터는 런타임이 끊겨도 몇 분 안에 복구됩니다.")
else:
    print("conda 환경 캐시가 이미 Drive에 있습니다:", DRIVE_CONDA_CACHE)

In [ ]:
# 5. 체크포인트 다운로드 (Drive에 캐싱해서 다음 실행부터는 재다운로드 안 함)
# download_weights.sh는 구버전 huggingface-cli / gdown --id 문법에 의존하는데,
# 최신 huggingface_hub(1.x)에서 huggingface-cli가 폐지되고 gdown도 --id를 제거해서
# 일부 파일이 에러 없이 조용히 다운로드되지 않습니다. 이 셀은 그걸 감지해서 안정적인 방식으로 직접 받습니다.
# 또한 실패한 다운로드 시도가 "빈 파일"이나 "잘린 파일"을 남길 수 있어서, 존재 여부뿐 아니라
# 최소 크기까지 확인해서 손상된 파일을 걸러내고 다시 받습니다.

# HF_TOKEN은 노트북에 직접 적어두지 않고 Colab Secrets(왼쪽 사이드바 열쇠 아이콘)에서 읽어옵니다.
# "HF_TOKEN"이라는 이름으로 등록해두고, 이 노트북에 대한 접근 권한을 켜주세요.
from google.colab import userdata

try:
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = ""

if HF_TOKEN and HF_TOKEN.startswith("hf_"):
    os.environ["HF_TOKEN"] = HF_TOKEN
    print("HF_TOKEN 설정됨 - 속도 제한 없이 다운로드합니다.")
else:
    print("HF_TOKEN 미설정 - 로그인 없이 받아서 느릴 수 있습니다.")
    print("왼쪽 사이드바 열쇠(Secrets) 아이콘에서 'HF_TOKEN' 이름으로 등록해주세요. (선택사항이지만 권장)")

DRIVE_CKPT_DIR = "/content/drive/MyDrive/musetalk_checkpoints"
LOCAL_MODELS_DIR = "/content/MuseTalk/models"
DOWNLOAD_PATH_ENV = f"{CONDA_PREFIX}/envs/{ENV_NAME}/bin:" + os.environ["PATH"]

%cd /content/MuseTalk

# 레포를 갓 클론하면 models 폴더 자체가 없어서, 아래 cp 명령이 "target is not a directory"로
# 조용히 실패하고 캐시 복원 자체가 안 되는 문제가 있었습니다. 미리 만들어둡니다.
os.makedirs(LOCAL_MODELS_DIR, exist_ok=True)

# (경로, 최소 정상 크기 bytes) - 다운로드가 중간에 끊기면 파일은 있어도 크기가 비정상적으로 작습니다.
# sd-vae, whisper도 download_weights.sh 안에서 huggingface-cli로 받는 항목이라 같은 문제가 생길 수 있어 포함시킵니다.
REQUIRED_FILES = [
    ("models/dwpose/dw-ll_ucoco_384.pth", 100_000_000),
    ("models/musetalkV15/unet.pth", 10_000_000),
    ("models/musetalkV15/musetalk.json", 10),
    ("models/face-parse-bisent/79999_iter.pth", 10_000_000),
    ("models/sd-vae/config.json", 10),
    ("models/sd-vae/diffusion_pytorch_model.bin", 100_000_000),
    ("models/whisper/config.json", 10),
    ("models/whisper/pytorch_model.bin", 10_000_000),
    ("models/whisper/preprocessor_config.json", 10),
]

def checkpoints_complete():
    for rel_path, min_size in REQUIRED_FILES:
        full_path = f"/content/MuseTalk/{rel_path}"
        if not os.path.exists(full_path) or os.path.getsize(full_path) < min_size:
            return False
    return True

def remove_broken_files():
    for rel_path, min_size in REQUIRED_FILES:
        full_path = f"/content/MuseTalk/{rel_path}"
        if os.path.exists(full_path) and os.path.getsize(full_path) < min_size:
            print(f"손상된(너무 작은) 파일 삭제: {full_path} ({os.path.getsize(full_path)} bytes)")
            os.remove(full_path)

if os.path.exists(DRIVE_CKPT_DIR) and os.listdir(DRIVE_CKPT_DIR):
    print("[캐시 발견] Drive에서 체크포인트 복사 중...")
    !cp -r "$DRIVE_CKPT_DIR"/* "$LOCAL_MODELS_DIR"/

remove_broken_files()

if not checkpoints_complete():
    print("기본 다운로드 스크립트 실행 (일부 파일은 다음 단계에서 직접 보완합니다)...")
    !PATH="{DOWNLOAD_PATH_ENV}" sh ./download_weights.sh
    remove_broken_files()

if not checkpoints_complete():
    print("누락 파일을 직접 받습니다: 먼저 wget으로 huggingface.co resolve URL을 시도하고,")
    print("실패하면 huggingface_hub 파이썬 API로 재시도(최대 3회, Xet 가속 백엔드는 꺼서 표준 HTTP로 받고 이어받기 가능)합니다.")
    fix_script = '''
import os
os.environ["HF_HUB_DISABLE_XET"] = "1"  # Xet 가속 백엔드가 간헐적으로 멈추는 이슈 회피, 표준 HTTP로 강제

import time, subprocess

hf_token = os.environ.get("HF_TOKEN")

targets = [
    ("yzd-v/DWPose", "dw-ll_ucoco_384.pth", "models/dwpose/dw-ll_ucoco_384.pth"),
    ("TMElyralab/MuseTalk", "musetalkV15/musetalk.json", "models/musetalkV15/musetalk.json"),
    ("TMElyralab/MuseTalk", "musetalkV15/unet.pth", "models/musetalkV15/unet.pth"),
    ("stabilityai/sd-vae-ft-mse", "config.json", "models/sd-vae/config.json"),
    ("stabilityai/sd-vae-ft-mse", "diffusion_pytorch_model.bin", "models/sd-vae/diffusion_pytorch_model.bin"),
    ("openai/whisper-tiny", "config.json", "models/whisper/config.json"),
    ("openai/whisper-tiny", "pytorch_model.bin", "models/whisper/pytorch_model.bin"),
    ("openai/whisper-tiny", "preprocessor_config.json", "models/whisper/preprocessor_config.json"),
]

def try_wget(repo_id, filename, dest):
    url = f"https://huggingface.co/{repo_id}/resolve/main/{filename}"
    cmd = ["wget", "-q", "-O", dest, url]
    if hf_token:
        cmd += ["--header", f"Authorization: Bearer {hf_token}"]
    result = subprocess.run(cmd)
    return result.returncode == 0 and os.path.exists(dest) and os.path.getsize(dest) > 0

def try_hf_hub_download(repo_id, filename, dest, attempts=3):
    from huggingface_hub import hf_hub_download
    import shutil
    for i in range(attempts):
        try:
            # force_download를 안 켜서, 중간에 끊겨도 다음 시도가 처음부터 다시 받지 않고 이어받습니다.
            path = hf_hub_download(repo_id=repo_id, filename=filename, token=hf_token)
            shutil.copy(path, dest)
            return True
        except Exception as e:
            print(f"  시도 {i+1}/{attempts} 실패: {e}")
            time.sleep(10)
    return False

for repo_id, filename, dest_rel in targets:
    dest = f"/content/MuseTalk/{dest_rel}"
    if os.path.exists(dest):
        continue
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    print(f"받는 중: {filename} ({repo_id})")
    if try_wget(repo_id, filename, dest):
        print("  wget으로 받음:", dest)
    elif try_hf_hub_download(repo_id, filename, dest):
        print("  huggingface_hub API로 받음:", dest)
    else:
        print("  받기 실패:", dest)
'''
    with open("/content/fix_downloads.py", "w") as f:
        f.write(fix_script)
    !{CONDA_PY} /content/fix_downloads.py
    remove_broken_files()

    face_parse_dest = "/content/MuseTalk/models/face-parse-bisent/79999_iter.pth"
    if not os.path.exists(face_parse_dest):
        os.makedirs(os.path.dirname(face_parse_dest), exist_ok=True)
        # gdown 최신 버전은 --id 옵션이 없어져서 파일 ID를 위치 인자로 바로 넘깁니다.
        !{CONDA_PY} -m gdown 154JgKpzCPW82qINcVieuPH3fZ2e0P812 -O {face_parse_dest}
    remove_broken_files()

# download_weights.sh / hf 관련 도구 설치 과정에서 huggingface_hub가 최신(1.x)으로 끌어올려지는 경우가 있습니다.
# transformers==4.39.2 / tokenizers==0.15.2는 huggingface_hub<1.0을 요구하므로, 다운로드가 끝난 뒤 다시 고정합니다.
!{CONDA_PIP} install -q "huggingface_hub==0.30.2"

if checkpoints_complete():
    os.makedirs(DRIVE_CKPT_DIR, exist_ok=True)
    !cp -r "$LOCAL_MODELS_DIR"/* "$DRIVE_CKPT_DIR"/
    print("모든 체크포인트 준비 완료 + Drive 백업 완료 (손상됐던 파일도 새로 백업되어 덮어써졌습니다)")
else:
    missing = [p for p, m in REQUIRED_FILES if not os.path.exists(f"/content/MuseTalk/{p}") or os.path.getsize(f"/content/MuseTalk/{p}") < m]
    print("여전히 누락/손상된 파일이 있습니다:", missing)

In [ ]:
# 6. 면접관 아바타 원본 영상 준비
# TODO: 파일명이 다르면 아래 경로를 실제 업로드한 파일명으로 수정하세요.
AVATAR_SOURCE_VIDEO = "/content/drive/MyDrive/musetalk_assets/avatar_source.mp4"

assert os.path.exists(AVATAR_SOURCE_VIDEO), f"영상을 찾을 수 없습니다: {AVATAR_SOURCE_VIDEO} - Drive 업로드 경로를 확인하세요."
print("아바타 원본 영상 확인 완료:", AVATAR_SOURCE_VIDEO)

# 레포를 갓 클론하면 data 폴더가 없어서 ffmpeg 출력이 실패하므로 미리 만들어둡니다.
os.makedirs("/content/MuseTalk/data", exist_ok=True)

# 참고: MuseTalk은 25fps 입력 기준으로 학습되었습니다.
!ffmpeg -y -i "$AVATAR_SOURCE_VIDEO" -r 25 /content/MuseTalk/data/avatar_source_25fps.mp4
AVATAR_SOURCE_VIDEO = "/content/MuseTalk/data/avatar_source_25fps.mp4"

In [ ]:
# 7. MuseTalk 서버 스크립트 작성
# 모델 로드 + 아바타 준비(Drive 캐싱) + FastAPI WebSocket 서버를 하나의 스크립트로 묶어서
# conda(Python 3.10) 환경의 파이썬으로 별도 프로세스 실행합니다.

server_script = '''
import os, sys, glob, time, types, shutil

# /content/MuseTalk 안의 musetalk 패키지(레포 내부 모듈)를 확실히 찾도록 경로를 명시적으로 추가합니다.
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))

import torch
from fastapi import FastAPI, WebSocket, WebSocketDisconnect
import uvicorn
from transformers import WhisperModel

from musetalk.utils.face_parsing import FaceParsing
from musetalk.utils.utils import load_all_model
from musetalk.utils.audio_processor import AudioProcessor

import scripts.realtime_inference as rt_module

# Avatar 클래스는 원래 커맨드라인(argparse)으로 실행되면서 만들어지는 여러 전역 변수
# (args, vae, unet, pe, timesteps, audio_processor, weight_dtype, whisper, fp)를 내부에서 참조합니다.
# 여기서는 그 실행 경로를 안 거치므로, 원본 __main__ 블록과 동일한 순서로 직접 만들어서 모듈에 주입합니다.
rt_module.args = types.SimpleNamespace(
    version="v15",
    ffmpeg_path="./ffmpeg-4.4-amd64-static/",
    gpu_id=0,
    vae_type="sd-vae",
    unet_config="./models/musetalkV15/musetalk.json",
    unet_model_path="./models/musetalkV15/unet.pth",
    whisper_dir="./models/whisper",
    inference_config="configs/inference/realtime.yaml",
    bbox_shift=0,
    result_dir="./results",
    extra_margin=10,
    fps=25,
    audio_padding_length_left=2,
    audio_padding_length_right=2,
    # A100은 원본 스크립트가 기준으로 삼은 V100보다 VRAM이 훨씬 넉넉해서(보통 40GB) 크게 올려봅니다.
    # OOM 나면 다시 낮추면 됩니다.
    batch_size=64,
    output_vid_name=None,
    use_saved_coord=False,
    saved_coord=False,
    parsing_mode="jaw",
    left_cheek_width=90,
    right_cheek_width=90,
    # skip_save_images=True는 "일부만 스킵"이 아니라 최종 mp4 자체를 만들지 않는
    # 순수 속도측정 전용 모드입니다. 실제 영상 결과물이 필요하므로 False로 둡니다.
    skip_save_images=False,
)
args = rt_module.args

device = torch.device(f"cuda:{args.gpu_id}" if torch.cuda.is_available() else "cpu")

print("모델 로딩 중...", flush=True)
vae, unet, pe = load_all_model(
    unet_model_path=args.unet_model_path,
    vae_type=args.vae_type,
    unet_config=args.unet_config,
    device=device,
)
timesteps = torch.tensor([0], device=device)
pe = pe.half().to(device)
vae.vae = vae.vae.half().to(device)
unet.model = unet.model.half().to(device)

audio_processor = AudioProcessor(feature_extractor_path=args.whisper_dir)
weight_dtype = unet.model.dtype
whisper = WhisperModel.from_pretrained(args.whisper_dir)
whisper = whisper.to(device=device, dtype=weight_dtype).eval()
whisper.requires_grad_(False)

if args.version == "v15":
    fp = FaceParsing(left_cheek_width=args.left_cheek_width, right_cheek_width=args.right_cheek_width)
else:
    fp = FaceParsing()

print("모델 로드 완료", flush=True)

# Avatar 클래스(scripts.realtime_inference 모듈)가 참조하는 전역들을 그대로 주입합니다.
rt_module.device = device
rt_module.vae = vae
rt_module.unet = unet
rt_module.pe = pe
rt_module.timesteps = timesteps
rt_module.audio_processor = audio_processor
rt_module.weight_dtype = weight_dtype
rt_module.whisper = whisper
rt_module.fp = fp

Avatar = rt_module.Avatar

AVATAR_ID = "interviewer_01"
DRIVE_AVATAR_DIR = f"/content/drive/MyDrive/musetalk_avatars/{AVATAR_ID}"
# v1.5는 내부적으로 "./results/v15/avatars/{avatar_id}" 경로를 씁니다 (버전별 하위 폴더).
LOCAL_AVATAR_DIR = f"/content/MuseTalk/results/v15/avatars/{AVATAR_ID}"
AVATAR_SOURCE_VIDEO = "/content/MuseTalk/data/avatar_source_25fps.mp4"

if os.path.exists(DRIVE_AVATAR_DIR):
    print("[캐시 발견] 아바타 데이터 Drive에서 복사", flush=True)
    os.makedirs(os.path.dirname(LOCAL_AVATAR_DIR), exist_ok=True)
    os.system(f'cp -r "{DRIVE_AVATAR_DIR}" "{LOCAL_AVATAR_DIR}"')
    preparation = False
else:
    print("[캐시 없음] 아바타 최초 준비 (시간 소요)", flush=True)
    preparation = True
    # 이전 실행이 중간에 실패해서 남긴 불완전한 폴더가 있으면, Avatar가 "다시 만들까요? (y/n)"라고
    # 대화형으로 물어보다가 (백그라운드 프로세스라 입력을 받을 수 없어) 죽습니다. 미리 깨끗하게 지웁니다.
    if os.path.exists(LOCAL_AVATAR_DIR):
        print("이전 미완성 아바타 데이터 삭제 중...", flush=True)
        shutil.rmtree(LOCAL_AVATAR_DIR)

avatar = Avatar(
    avatar_id=AVATAR_ID,
    video_path=AVATAR_SOURCE_VIDEO,
    bbox_shift=0,
    batch_size=64,
    preparation=preparation,
)
if preparation:
    avatar.prepare_material()
    os.makedirs(os.path.dirname(DRIVE_AVATAR_DIR), exist_ok=True)
    os.system(f'cp -r "{LOCAL_AVATAR_DIR}" "{DRIVE_AVATAR_DIR}"')
    print("아바타 준비 결과 Drive 백업 완료", flush=True)

OUTPUT_DIR = f"{LOCAL_AVATAR_DIR}/vid_output"

app = FastAPI()

@app.websocket("/synthesize")
async def synthesize(websocket: WebSocket):
    await websocket.accept()
    turn = 0
    try:
        while True:
            audio_bytes = await websocket.receive_bytes()
            turn += 1
            audio_path = f"/content/tmp_turn_{turn}.wav"
            with open(audio_path, "wb") as f:
                f.write(audio_bytes)

            out_name = f"turn_{turn}"
            t0 = time.time()
            avatar.inference(
                audio_path=audio_path,
                out_vid_name=out_name,
                fps=25,
                skip_save_images=False,
            )
            elapsed = time.time() - t0

            candidates = glob.glob(f"{OUTPUT_DIR}/{out_name}*.mp4")
            if not candidates:
                await websocket.send_json({"error": f"출력 영상을 찾지 못했습니다: {OUTPUT_DIR}/{out_name}*.mp4"})
                continue

            with open(candidates[0], "rb") as f:
                video_bytes = f.read()

            print(f"[turn {turn}] 추론 {elapsed:.2f}초, 영상 {len(video_bytes)/1024:.1f}KB", flush=True)
            await websocket.send_bytes(video_bytes)
    except WebSocketDisconnect:
        print("클라이언트 연결 종료", flush=True)

if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="info")
'''

with open("/content/MuseTalk/musetalk_server.py", "w", encoding="utf-8") as f:
    f.write(server_script)

print("musetalk_server.py 작성 완료")

## 서버 기동

모델 로딩 + (최초 1회) 아바타 준비까지 포함되어 있어서 처음 실행은 몇 분 걸릴 수 있습니다.
conda 환경의 파이썬으로 별도 프로세스를 띄우고, 로그에 `Uvicorn running on`이 찍힐 때까지 기다립니다.

In [ ]:
# 8. 서버 프로세스 실행 + 기동 대기
import subprocess, time

# 이 셀을 재실행할 때 이전에 띄워둔 서버 프로세스가 아직 살아있으면(8000번 포트 점유),
# 새 프로세스가 포트 충돌로 기동에 실패하므로 먼저 정리합니다.
if "server_proc" in globals() and server_proc.poll() is None:
    print("이전 서버 프로세스 종료 중...")
    server_proc.kill()
    server_proc.wait()

log_path = "/content/musetalk_server.log"
log_file = open(log_path, "w")

# 노트북 커널의 MPLBACKEND(matplotlib_inline)가 그대로 상속되면 conda 환경(3.10)에
# 해당 패키지가 없어 matplotlib import 시 에러가 나므로, 서버 프로세스용으로 headless 백엔드로 덮어씁니다.
server_env = dict(os.environ)
server_env["MPLBACKEND"] = "Agg"

server_proc = subprocess.Popen(
    [CONDA_PY, "/content/MuseTalk/musetalk_server.py"],
    cwd="/content/MuseTalk",
    stdout=log_file,
    stderr=subprocess.STDOUT,
    env=server_env,
)
print(f"서버 프로세스 시작 (PID {server_proc.pid})")

content = ""
ready = False
for _ in range(120):  # 최대 10분 대기 (모델 로딩 + 최초 아바타 준비 포함)
    with open(log_path) as f:
        content = f.read()
    if "Uvicorn running on" in content:
        ready = True
        break
    if server_proc.poll() is not None:
        print("서버 프로세스가 예기치 않게 종료되었습니다. 아래 로그를 확인하세요.")
        break
    time.sleep(5)

print("서버 기동 완료!" if ready else "아직 기동되지 않았습니다 - 이 셀을 다시 실행해서 계속 대기하거나 로그를 확인하세요.")
print("----- 최근 로그 -----")
print(content[-3000:])

## 순수 추론 속도 벤치마크 (로컬 루프백)

ngrok으로 외부에 공개하기 전에, 같은 머신 안에서(`localhost`) 오디오를 보내 왕복 시간을 먼저 확인합니다.
여기서 나온 시간이 MuseTalk 자체 처리 속도의 상한선입니다 (ngrok을 거치면 네트워크 왕복이 추가로 붙습니다).

In [ ]:
# 9. 로컬 루프백 벤치마크 (텍스트 → Edge TTS 음성 → MuseTalk 영상)
# 실제 서비스에서 MuseTalk에 들어가는 오디오는 항상 TTS로 만들어진 것이므로,
# sample.wav를 찾는 대신 텍스트를 직접 음성으로 변환해서 테스트합니다.
!pip install -q edge-tts

TEST_TEXT = "안녕하세요, 반갑습니다. 오늘 면접에 참여해주셔서 감사합니다."
TTS_MP3_PATH = "/content/tts_test.mp3"
TEST_AUDIO_PATH = "/content/tts_test.wav"

!edge-tts --voice ko-KR-SunHiNeural --text "{TEST_TEXT}" --write-media {TTS_MP3_PATH}
!ffmpeg -y -i {TTS_MP3_PATH} {TEST_AUDIO_PATH}

import asyncio, websockets, time

async def benchmark():
    with open(TEST_AUDIO_PATH, "rb") as f:
        audio_bytes = f.read()
    async with websockets.connect("ws://localhost:8000/synthesize", max_size=None) as ws:
        t0 = time.time()
        await ws.send(audio_bytes)
        result = await ws.recv()
        elapsed = time.time() - t0

        if isinstance(result, str):
            # 서버가 영상 대신 에러 메시지(JSON 텍스트)를 보낸 경우
            print(f"서버가 에러를 반환했습니다 (왕복 {elapsed:.2f}초):")
            print(result)
            return

        out_path = "/content/benchmark_result.mp4"
        with open(out_path, "wb") as f:
            f.write(result)
        print(f"왕복 시간: {elapsed:.2f}초, 받은 영상 크기: {len(result)/1024:.1f}KB")
        print(f"결과 영상 저장 위치: {out_path} (Colab 왼쪽 파일 탐색기에서 다운로드해 확인 가능)")

await benchmark()

In [ ]:
# 10. ngrok 터널 오픈 (고정 도메인)
# NGROK_AUTHTOKEN / NGROK_STATIC_DOMAIN도 노트북에 직접 적지 않고 Colab Secrets에서 읽어옵니다.
# 왼쪽 사이드바 열쇠(Secrets) 아이콘에 아래 이름 그대로 등록해주세요.
#   NGROK_AUTHTOKEN     - https://dashboard.ngrok.com/get-started/your-authtoken 에서 발급
#   NGROK_STATIC_DOMAIN - https://dashboard.ngrok.com/domains 에서 예약 (예: your-name.ngrok-free.app)
from pyngrok import ngrok, conf
from google.colab import userdata

NGROK_AUTHTOKEN = userdata.get("NGROK_AUTHTOKEN")
NGROK_STATIC_DOMAIN = userdata.get("NGROK_STATIC_DOMAIN")

conf.get_default().auth_token = NGROK_AUTHTOKEN

public_tunnel = ngrok.connect(8000, domain=NGROK_STATIC_DOMAIN)
print("MuseTalk 서버 공개 URL:", public_tunnel.public_url)
print("WebSocket 엔드포인트:", public_tunnel.public_url.replace("https://", "wss://") + "/synthesize")

## 백엔드 연결

위 셀에서 출력된 `wss://...ngrok-free.app/synthesize` 값을 로컬 `backend/.env`에 다음과 같이 넣으세요:

```
MUSETALK_SERVICE_URL=wss://your-name.ngrok-free.app/synthesize
```

고정 도메인을 썼다면 Colab 런타임을 재시작해도 이 값은 그대로 유효합니다 (노트북을 다시 실행해서 서버만 재기동하면 됨).